# Data Collection

This notebook downloads the [Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii) dataset from the UCI Machine Learning Repository and saves it to S3 as a parquet file. The dataset contains transactional data from a UK-based online retail company (Dec 2009 - Dec 2011) and will be used for customer segmentation via cluster analysis.

## Imports

In [1]:
import pandas as pd
import zipfile
import io
import requests
import pyarrow
import s3fs
try:
    import openpyxl
except:
    ! pip install openpyxl
    import openpyxl

## Constants

In [2]:
str_bucket = 'cluster-analysis-demo'
str_task = '00_data_collection'
str_url = 'https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip'

## Download Data

In [3]:
# download zip file
response = requests.get(str_url)
response.raise_for_status()

# extract xlsx from zip
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    # find the xlsx file
    list_files = z.namelist()
    print(f'Files in zip: {list_files}')
    # get xlsx filename
    str_xlsx = [f for f in list_files if f.endswith('.xlsx')][0]
    # read xlsx
    with z.open(str_xlsx) as f:
        df = pd.read_excel(f, engine='openpyxl')

Files in zip: ['online_retail_II.xlsx']


## Inspect Data

In [4]:
# shape
print(f'Shape: {df.shape}')

# data types
print(f'\nData Types:\n{df.dtypes}')

# first 5 rows
df.head()

Shape: (525461, 8)

Data Types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
# missing values
print(f'Missing Values:\n{df.isnull().sum()}')
print(f'\nMissing Value Proportion: {df.isnull().sum().sum() / (df.shape[0] * df.shape[1]):.4f}')

Missing Values:
Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

Missing Value Proportion: 0.0264


In [6]:
# unique customers
print(f'Unique Invoices: {df["Invoice"].nunique():,}')
print(f'Unique Customers: {df["Customer ID"].nunique():,}')
print(f'Unique Products: {df["StockCode"].nunique():,}')
print(f'Unique Countries: {df["Country"].nunique():,}')
print(f'\nDate Range: {df["InvoiceDate"].min()} to {df["InvoiceDate"].max()}')

Unique Invoices: 28,816
Unique Customers: 4,383
Unique Products: 4,632
Unique Countries: 40

Date Range: 2009-12-01 07:45:00 to 2010-12-09 20:01:00


## Clean Column Names

In [7]:
# clean column names (snake_case)
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w]', '', regex=True)
)

# cast object columns to string (mixed int/str causes Arrow errors)
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str)

# check
print(f'Columns: {df.columns.tolist()}')
print(f'Dtypes:\n{df.dtypes}')

Columns: ['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country']
Dtypes:
invoice                object
stockcode              object
description            object
quantity                int64
invoicedate    datetime64[ns]
price                 float64
customer_id           float64
country                object
dtype: object


## Save to S3

In [8]:
# save to s3 as parquet
str_uri = f's3://{str_bucket}/{str_task}/data.parquet'
df.to_parquet(str_uri, index=False)
print(f'Saved {df.shape[0]:,} rows to {str_uri}')

Saved 525,461 rows to s3://cluster-analysis-demo/00_data_collection/data.parquet
